<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Behavioral_cloning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [29]:
!unzip /content/archive.zip -d "/content/"

Archive:  /content/archive.zip
replace /content/IMG/center_2021_12_19_18_46_10_430.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [67]:
!find /content -name driving_log.csv

/content/data/driving_log.csv
/content/driving_log.csv


In [80]:
df = pd.read_csv("/content/driving_log.csv", header=None)
df.columns = ["center", "left", "right", "steering", "throttle", "brake", "speed"]

# Steering angle column
steering = df["steering"].values

# Image path column
image_paths = df["center"].values

print("Rows:", len(df))
df

Rows: 1795


,center,left,right,steering,throttle,brake,speed
0,/Users/asik/Desktop/Self Driving Car/IMG/cente...,/Users/asik/Desktop/Self Driving Car/IMG/left...,/Users/asik/Desktop/Self Driving Car/IMG/righ...,0.0,0.0,0.0,0.000079
1,/Users/asik/Desktop/Self Driving Car/IMG/cente...,/Users/asik/Desktop/Self Driving Car/IMG/left...,/Users/asik/Desktop/Self Driving Car/IMG/righ...,0.0,0.0,0.0,0.000079
2,/Users/asik/Desktop/Self Driving Car/IMG/cente...,/Users/asik/Desktop/Self Driving Car/IMG/left...,/Users/asik/Desktop/Self Driving Car/IMG/righ...,0.0,0.0,0.0,0.000080
3,/Users/asik/Desktop/Self Driving Car/IMG/cente...,/Users/asik/Desktop/Self Driving Car/IMG/left...,/Users/asik/Desktop/Self Driving Car/IMG/righ...,0.0,0.0,0.0,0.000080
4,/Users/asik/Desktop/Self Driving Car/IMG/cente...,/Users/asik/Desktop/Self Driving Car/IMG/left...,/Users/asik/Desktop/Self Driving Car/IMG/righ...,0.0,0.0,0.0,0.000079
...,...,...,...,...,...,...,...
1790,/Users/asik/Desktop/Self Driving Car/IMG/cente...,/Users/asik/Desktop/Self Driving Car/IMG/left...,/Users/asik/Desktop/Self Driving Car/IMG/righ...,0.0,0.0,0.0,2.129716
1791,/Users/asik/Desktop/Self Driving Car/IMG/cente...,/Users/asik/Desktop/Self Driving Car/IMG/left...,/Users/asik/Desktop/Self Driving Car/IMG/righ...,0.0,0.0,0.0,2.101073
1792,/Users/asik/Desktop/Self Driving Car/IMG/cente...,/Users/asik/Desktop/Self Driving Car/IMG/left...,/Users/asik/Desktop/Self Driving Car/IMG/righ...,0.0,0.0,0.0,2.069010
1793,/Users/asik/Desktop/Self Driving Car/IMG/cente...,/Users/asik/Desktop/Self Driving Car/IMG/left...,/Users/asik/Desktop/Self Driving Car/IMG/righ...,0.0,0.0,0.0,2.037461


In [81]:
df["center"] = df["center"].apply(lambda x: x.strip().split("/")[-1])
df["left"]   = df["left"].apply(lambda x: x.strip().split("/")[-1])
df["right"]  = df["right"].apply(lambda x: x.strip().split("/")[-1])


In [82]:
DATA_PATH = "IMG/"

In [83]:
from os import listdir
print(listdir(DATA_PATH))

['left_2021_12_19_18_46_10_430.jpg', 'right_2021_12_19_18_46_10_430.jpg', 'center_2021_12_19_18_46_10_430.jpg']


In [72]:
def load_image(filename):
    path = DATA_PATH + filename
    img = cv2.imread(path)

    # Debug if something fails
    if img is None:
        print("Could not load:", path)
        return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (200,66))
    return img

In [73]:
df["center"] = df["center"].apply(lambda x: x.split("/")[-1])
df["left"]   = df["left"].apply(lambda x: x.split("/")[-1])
df["right"]  = df["right"].apply(lambda x: x.split("/")[-1])

In [74]:
print(df["center"].iloc[0])

center_2021_12_19_18_46_10_430.jpg


In [75]:
import os
print(os.listdir(DATA_PATH)[:5])

['left_2021_12_19_18_46_10_430.jpg', 'right_2021_12_19_18_46_10_430.jpg', 'center_2021_12_19_18_46_10_430.jpg']


In [84]:
test = load_image(df["center"].iloc[0])
print(test)

[[[116 145 185]
  [116 145 185]
  [115 144 184]
  ...
  [ 20  29  16]
  [ 18  26  11]
  [ 12  22   9]]

 [[118 147 187]
  [118 147 187]
  [117 147 187]
  ...
  [ 15  19   7]
  [ 23  29  15]
  [ 30  36  21]]

 [[121 150 190]
  [121 150 190]
  [120 149 189]
  ...
  [ 47  48  30]
  [ 80  82  63]
  [ 38  39  19]]

 ...

 [[125 130 124]
  [ 99 104  97]
  [ 92  97  91]
  ...
  [ 99 100  84]
  [ 99 100  84]
  [109 110  94]]

 [[123 128 122]
  [ 88  93  87]
  [ 92  97  91]
  ...
  [140 141 125]
  [107 108  93]
  [108 109  93]]

 [[ 63  68  62]
  [ 66  71  65]
  [ 73  78  72]
  ...
  [125 126 110]
  [103 104  88]
  [ 86  87  71]]]


In [85]:
images = []
angles = []
correction = 0.2

for i, row in df.iterrows():
    steering = float(row["steering"])

    # center
    img_center = load_image(row["center"])
    if img_center is not None:
        images.append(img_center)
        angles.append(steering)

    # left
    img_left = load_image(row["left"])
    if img_left is not None:
        images.append(img_left)
        angles.append(steering + correction)

    # right
    img_right = load_image(row["right"])
    if img_right is not None:
        images.append(img_right)
        angles.append(steering - correction)


Streaming output truncated to the last 5000 lines.
Could not load: IMG/left_2021_12_19_18_46_27_053.jpg
Could not load: IMG/right_2021_12_19_18_46_27_053.jpg
Could not load: IMG/center_2021_12_19_18_46_27_336.jpg
Could not load: IMG/left_2021_12_19_18_46_27_336.jpg
Could not load: IMG/right_2021_12_19_18_46_27_336.jpg
Could not load: IMG/center_2021_12_19_18_46_27_462.jpg
Could not load: IMG/left_2021_12_19_18_46_27_462.jpg
Could not load: IMG/right_2021_12_19_18_46_27_462.jpg
Could not load: IMG/center_2021_12_19_18_46_27_593.jpg
Could not load: IMG/left_2021_12_19_18_46_27_593.jpg
Could not load: IMG/right_2021_12_19_18_46_27_593.jpg
Could not load: IMG/center_2021_12_19_18_46_27_725.jpg
Could not load: IMG/left_2021_12_19_18_46_27_725.jpg
Could not load: IMG/right_2021_12_19_18_46_27_725.jpg
Could not load: IMG/center_2021_12_19_18_46_27_857.jpg
Could not load: IMG/left_2021_12_19_18_46_27_857.jpg
Could not load: IMG/right_2021_12_19_18_46_27_857.jpg
Could not load: IMG/center_2021_

In [87]:
print(len(df))

1795


In [86]:
X = np.array(images)
y = np.array(angles)

print("Dataset:", X.shape, y.shape)

Dataset: (3, 66, 200, 3) (3,)


In [66]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=True)